# 07b · Nulo de estructura constante y lectura de la parte dinámica (vigente: niveles, 16 series)

**Objetivo.** Primera pieza de OE11 (y de HT1): una distribución de referencia para los $k^*$ de
`aede` y para el coeficiente RV entre ventanas, bajo la hipótesis de que la estructura de
correlaciones **no cambia** en el tiempo (tarea D7 del plan), y su lectura con los criterios de
D10, D13 y D17 (tarea D10). Toda lectura de un $k^*$ o de un corte se hace contra este nulo, nunca
contra los episodios (regla 6).

**Construcción** (remuestreo por bloques **conjunto**, $L = 24$; sensibilidad con $L = 12$ y
$L = 36$): los mismos inicios de bloque para todas las series. En niveles se remuestrean los
incrementos mensuales y se acumulan desde 0 (versión «con deriva» del nulo de tendencias). Se
conservan la correlación contemporánea y la dinámica corta y se dispersa en el tiempo cualquier
cambio de estructura. *Supuesto declarado:* el nulo acumula incrementos desde 0 y no es
estacionario en el tiempo; el bloque conjunto mezcla tramos de regímenes distintos (riesgo 2).

**Medidas** (`src/nulo_estructura.py`), para cada réplica, ancho $n_w$ ∈ {24, 36, 48, 60} y pesos
(uniformes, gaussianos `nsd = 3`):

1. el $k^*$ de cada criterio de `aede` con maxcl = 20;
2. el perfil del RV entre **ventanas adyacentes no solapadas** (la que empieza en la fila $i$ y la
   que empieza en $i + n_w$), con los dos objetos de **D17.2**: las cargas de `aede` ($p \times 3$)
   **sin centrar** (principal) y las matrices de correlación completas $p \times p$ con los pesos
   de `aede` (sensibilidad).

**Criterio de OE4 (D17.2).** El p5 del nulo se toma **por posición** (cada par tiene su propio p5
entre las réplicas). El estadístico es la proporción de pares observados bajo su p5, y se compara
con la distribución nula de esa misma proporción, en la que el p5 de cada réplica se calcula
**dejándola fuera** (`src/dinamica.criterio_oe4`). La especificación principal es $L = 24$ con 200
réplicas y las cargas sin centrar.

**Réplicas y paridad (D6).** El guion `R/07b_nulo_estructura.R` sortea 200 réplicas por $L$
(semilla 2026) y calcula las 20 primeras; este notebook usa las mismas réplicas (`SEMILLA = None`).
Las réplicas 21 a 200 con $L = 24$ se calculan **solo en Python** (excepción declarada en la tabla de
paridad del Rmd), y el criterio de OE4 se compara con R en su versión de 20 réplicas.

**Lectura de D10, D13 y D17.3** (sección 4, solo niveles). Transiciones de la regla del 06 y de
`aede` (codo, maxcl = 10) con la marca de borde de ventana (D7, ±3 meses; ±6 como sensibilidad) y
su posición frente al nulo, y la tabla de 2017-10 con la regla de D13 leída según D17.3.

**Salidas:** `07b_nulo_estructura_{kstar,kstar_posicion,rv,rv_resumen,rv_nulo,rv_posicion,oe4}.csv`,
`07b_transiciones_d10.csv`, `07b_tabla_2017_10.csv`, `07b_clasificacion_2017_10.csv` y
`07b_bordes_ventana.csv`.

In [1]:
# ---- Configuración y parámetros (adenda 3b, §3) ------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src.parametros import leer_parametros
from src.datos import cargar_vias, meta_vigente

PARAM = leer_parametros(RAIZ)
VIA, EXCLUIDAS, DIR, DIR_R = PARAM['VIA'], PARAM['EXCLUIDAS'], PARAM['DIR'], PARAM['DIR_R']
ANTECEDENTE = len(EXCLUIDAS) == 0          # 17 series: modo de no regresión
X, Xt = cargar_vias(RAIZ, excluidas=EXCLUIDAS)
META = meta_vigente(EXCLUIDAS)
P = X.shape[1]
TODAS = {'niveles': X, 'transformadas': Xt}
VIAS = {v: (TODAS[v], DIR[v], DIR_R[v]) for v in PARAM['VIAS_ACTIVAS']}
print('VIA =', VIA, '| EXCLUIDAS =', EXCLUIDAS, '| p =', P, '| CARPETA =', PARAM['CARPETA'])
for via, (M, d, dr) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1], '| salidas:', d.relative_to(RAIZ) if d.is_relative_to(RAIZ) else d)

import time
from src import nulo_estructura as ne
from src import dinamica as dn
from src.episodios import cargar_episodios, bordes_ventana
from src.remuestreo import leer_inicios_bootstrap

SEMILLA = None
REPLICAS = {12: 20, 24: 200, 36: 20}      # réplicas por longitud de bloque
B_PARIDAD = 20
OBJETOS = ne.OBJETOS_D17
MAXCL_AEDE = 10                           # D17.1: especificación principal para las transiciones
INICIOS = {}
for via, (M, d, dr) in VIAS.items():
    INICIOS[via] = {L: leer_inicios_bootstrap(dr / f'R_nulo_estructura_indices_L{L}.csv') for L in REPLICAS}
    for L, ini in INICIOS[via].items():
        assert ini.shape[0] >= REPLICAS[L] and ini.min() >= 1 and ini.max() <= 227 - L + 1
        print(f'{via}: L = {L}: {ini.shape[0]} réplicas sorteadas, {ini.shape[1]} bloques; se usan {REPLICAS[L]}')

VIA = niveles | EXCLUIDAS = ('Liquidez M2',) | p = 16 | CARPETA = /home/claude/tesis-modelamiento-matematico/data/results/final
niveles (228, 16) 2007-01 → 2025-12 | salidas: data/results/final


niveles: L = 12: 200 réplicas sorteadas, 19 bloques; se usan 20
niveles: L = 24: 200 réplicas sorteadas, 10 bloques; se usan 200
niveles: L = 36: 200 réplicas sorteadas, 7 bloques; se usan 20


## 1. Serie observada y réplicas nulas

In [2]:
resultados = {}
t0 = time.time()
for via, (M, d, dr) in VIAS.items():
    k0, rv0, rr0 = ne.medir(M, objetos_rv=OBJETOS)
    kst = [k0.assign(L=0, replica=0)]
    rvp = [rv0.assign(replica=0)]
    rvr = [rr0.assign(L=0, replica=0)]
    perfiles = {}                                    # (L, objeto, nw, pesos) -> lista de perfiles por réplica
    for L, B in REPLICAS.items():
        for b in range(1, B + 1):
            k, rv, rr = ne.medir(ne.serie_nula(M, via, INICIOS[via][L][b - 1], L), objetos_rv=OBJETOS)
            kst.append(k.assign(L=L, replica=b))
            rvr.append(rr.assign(L=L, replica=b))
            if L == 24 and b <= B_PARIDAD:
                rvp.append(rv.assign(replica=b))
            for (obj, nw, pz), g in rv.groupby(['objeto', 'nw', 'pesos'], sort=False):
                perfiles.setdefault((L, obj, nw, pz), []).append(g.rv.to_numpy())
        print(f'{via}: L = {L}, {B} réplicas ({(time.time() - t0) / 60:.1f} min)')
    resultados[via] = {'kstar': pd.concat(kst, ignore_index=True), 'rv': pd.concat(rvp, ignore_index=True),
                       'rv_resumen': pd.concat(rvr, ignore_index=True), 'perfiles': perfiles, 'rv_obs': rv0}

niveles: L = 12, 20 réplicas (0.6 min)


niveles: L = 24, 200 réplicas (6.4 min)


niveles: L = 36, 20 réplicas (7.0 min)


## 2. Tablas de salida: posición de cada $k^*$ y criterio de OE4

`prop_nulo_ge_obs` es la proporción de réplicas con $k^*$ mayor o igual que el observado y
`prop_nulo_le_obs`, con $k^*$ menor o igual; `prop_nulo_en_tope`, la proporción de réplicas cuyo
criterio elige el tope (k = 20). El $k^*$ observado con maxcl = 20 es el de `06c_aede_cortes.csv`.

En `07b_nulo_estructura_oe4.csv`, la fila principal (`principal = True`) es $L = 24$, 200 réplicas y
cargas sin centrar. Las filas con 20 réplicas son las de paridad con R (mismas réplicas).

In [3]:
CONJUNTOS = [(24, 200), (24, B_PARIDAD), (12, 20), (36, 20)]     # (L, réplicas) del criterio de OE4
for via, (M, d, dr) in VIAS.items():
    r = resultados[via]
    ks = r['kstar'][['L', 'replica', 'nw', 'pesos', 'criterio', 'k', 'k_en_tope']].copy()
    ks['en_paridad'] = ks.replica <= B_PARIDAD
    ks.to_csv(d / '07b_nulo_estructura_kstar.csv', index=False)
    r['rv'][['replica', 'objeto', 'nw', 'pesos', 'fecha_a', 'fecha_b', 'frontera', 'rv']].to_csv(d / '07b_nulo_estructura_rv.csv', index=False)
    rr = r['rv_resumen'][['L', 'replica', 'objeto', 'nw', 'pesos', 'n_pares', 'rv_min', 'rv_p05', 'rv_p50', 'rv_media']].copy()
    rr['en_paridad'] = rr.replica <= B_PARIDAD
    rr.to_csv(d / '07b_nulo_estructura_rv_resumen.csv', index=False)
    # Cuantiles del RV nulo agrupado (todas las posiciones de todas las réplicas de cada L): descriptivo
    filas = []
    for (L, obj, nw, pz), lst in sorted(r['perfiles'].items()):
        v = np.concatenate(lst)
        filas.append({'L': L, 'objeto': obj, 'nw': nw, 'pesos': pz, 'replicas': len(lst), 'n_valores': len(v),
                      'p01': np.quantile(v, 0.01), 'p05': np.quantile(v, 0.05), 'p10': np.quantile(v, 0.10),
                      'p50': np.quantile(v, 0.50)})
    pd.DataFrame(filas).to_csv(d / '07b_nulo_estructura_rv_nulo.csv', index=False)
    # Posición del k* observado (réplica 0) en la distribución nula de cada L
    obs = ks[ks.replica == 0].set_index(['nw', 'pesos', 'criterio']).k
    g06c = pd.read_csv(d / '06c_aede_cortes.csv')
    g06c = g06c[g06c.maxcl == 20].set_index(['nw', 'pesos', 'criterio']).k
    assert (obs.sort_index() == g06c.sort_index()).all(), 'el k* observado no coincide con 06c (maxcl = 20)'
    pos = []
    for L in REPLICAS:
        nul = ks[(ks.L == L) & (ks.replica > 0)]
        for (nw, pz, cr), g in nul.groupby(['nw', 'pesos', 'criterio'], sort=False):
            k_obs = int(obs.loc[(nw, pz, cr)])
            pos.append({'L': L, 'nw': nw, 'pesos': pz, 'criterio': cr, 'k_obs': k_obs, 'k_obs_en_tope': k_obs == ne.MAXCL,
                        'replicas': len(g), 'nulo_p05': np.quantile(g.k, 0.05), 'nulo_p50': np.quantile(g.k, 0.5),
                        'nulo_p95': np.quantile(g.k, 0.95), 'prop_nulo_ge_obs': float(np.mean(g.k >= k_obs)),
                        'prop_nulo_le_obs': float(np.mean(g.k <= k_obs)), 'prop_nulo_en_tope': float(np.mean(g.k_en_tope))})
    pd.DataFrame(pos).to_csv(d / '07b_nulo_estructura_kstar_posicion.csv', index=False)
    # Criterio de OE4 (D17.2): p5 por posición y proporción de pares bajo el p5 frente a su nulo
    oe4, posicion = [], []
    for L, B in CONJUNTOS:
        for obj in OBJETOS:
            for nw in ne.NW_GRILLA:
                for pz in ne.PESOS:
                    o = r['rv_obs'][(r['rv_obs'].objeto == obj) & (r['rv_obs'].nw == nw) & (r['rv_obs'].pesos == pz)]
                    R = np.vstack(r['perfiles'][(L, obj, nw, pz)][:B])
                    c = dn.criterio_oe4(o.rv.to_numpy(), R)
                    oe4.append({'L': L, 'replicas': B, 'objeto': obj, 'nw': nw, 'pesos': pz, 'n_pares': len(o),
                                'pares_bajo_p5': int(c['bajo'].sum()), 'prop_obs': c['prop_obs'],
                                'prop_nulo_p50': float(np.quantile(c['props_nulo'], 0.5)),
                                'prop_nulo_p95': float(np.quantile(c['props_nulo'], 0.95)), 'p_valor': c['p_valor'],
                                'principal': L == 24 and B == 200 and obj == dn.OBJETO_PRINCIPAL})
                    posicion.append(pd.DataFrame({'L': L, 'replicas': B, 'objeto': obj, 'nw': nw, 'pesos': pz,
                                                  'fecha_a': o.fecha_a.to_numpy(), 'fecha_b': o.fecha_b.to_numpy(),
                                                  'frontera': o.frontera.to_numpy(), 'rv_obs': o.rv.to_numpy(),
                                                  'p5': c['p5'], 'bajo_p5': c['bajo']}))
    oe4 = pd.DataFrame(oe4)
    oe4.to_csv(d / '07b_nulo_estructura_oe4.csv', index=False)
    pd.concat(posicion, ignore_index=True).to_csv(d / '07b_nulo_estructura_rv_posicion.csv', index=False)
    resultados[via]['oe4'] = oe4
    print(via)
    display(oe4[(oe4.L == 24) & (oe4.replicas == 200)])

niveles


,L,replicas,objeto,nw,pesos,n_pares,pares_bajo_p5,prop_obs,prop_nulo_p50,prop_nulo_p95,p_valor,principal
0,24,200,cargas_sin_centrar,24,uniformes,181,4,0.022099,0.044199,0.154696,0.695,True
1,24,200,cargas_sin_centrar,24,gaussianos_nsd3,181,6,0.033149,0.044199,0.154972,0.620,True
2,24,200,cargas_sin_centrar,36,uniformes,157,8,0.050955,0.038217,0.192038,0.450,True
3,24,200,cargas_sin_centrar,36,gaussianos_nsd3,157,10,0.063694,0.031847,0.197452,0.380,True
4,24,200,cargas_sin_centrar,48,uniformes,133,0,0.000000,0.011278,0.203383,1.000,True
5,24,200,cargas_sin_centrar,48,gaussianos_nsd3,133,2,0.015038,0.003759,0.219549,0.475,True
6,24,200,cargas_sin_centrar,60,uniformes,109,0,0.000000,0.000000,0.238991,1.000,True
7,24,200,cargas_sin_centrar,60,gaussianos_nsd3,109,0,0.000000,0.000000,0.220642,1.000,True
8,24,200,correlacion,24,uniformes,181,8,0.044199,0.044199,0.154972,0.545,False
9,24,200,correlacion,24,gaussianos_nsd3,181,5,0.027624,0.044199,0.149448,0.690,False


## 3. Comparación con R (20 réplicas por longitud de bloque y serie observada)

$k^*$ exactos; perfiles RV ($L = 24$), resúmenes por réplica, p5 por posición y criterio de OE4 con
20 réplicas, con tolerancia $10^{-10}$ (marcas bajo el p5, exactas).

In [4]:
for via, (M, d, dr) in VIAS.items():
    f_k = dr / 'R_nulo_estructura_kstar.csv'
    if not f_k.exists():
        print(f'{via}: aún no existe {f_k.name} (ejecutar R/07b_nulo_estructura.R)')
        continue
    rk = pd.read_csv(f_k)
    pk = pd.read_csv(d / '07b_nulo_estructura_kstar.csv')
    clave = ['L', 'replica', 'nw', 'pesos', 'criterio']
    m = pk.merge(rk, on=clave, suffixes=('', '_r'))
    assert len(m) == len(rk), (via, len(m), len(rk))
    n_dif = int((m.k != m.k_r).sum())
    cl_rv = ['replica', 'objeto', 'nw', 'pesos', 'fecha_a', 'fecha_b', 'frontera']
    pr = pd.read_csv(d / '07b_nulo_estructura_rv.csv').merge(pd.read_csv(dr / 'R_nulo_estructura_rv.csv'), on=cl_rv, suffixes=('', '_r'))
    dif_rv = float(np.max(np.abs(pr.rv - pr.rv_r)))
    rs = pd.read_csv(d / '07b_nulo_estructura_rv_resumen.csv').merge(pd.read_csv(dr / 'R_nulo_estructura_rv_resumen.csv'),
                                                                     on=['L', 'replica', 'objeto', 'nw', 'pesos'], suffixes=('', '_r'))
    dif_rs = max(float(np.max(np.abs(rs[c] - rs[c + '_r']))) for c in ['rv_min', 'rv_p05', 'rv_p50', 'rv_media'])
    po = pd.read_csv(d / '07b_nulo_estructura_rv_posicion.csv')
    po = po[po.replicas == B_PARIDAD].merge(pd.read_csv(dr / 'R_nulo_estructura_rv_posicion.csv'),
                                            on=['L', 'replicas', 'objeto', 'nw', 'pesos', 'fecha_a', 'fecha_b', 'frontera'], suffixes=('', '_r'))
    dif_p5 = float(np.max(np.abs(po.p5 - po.p5_r)))
    n_marca = int((po.bajo_p5 != po.bajo_p5_r).sum())
    oe = pd.read_csv(d / '07b_nulo_estructura_oe4.csv')
    oe = oe[oe.replicas == B_PARIDAD].merge(pd.read_csv(dr / 'R_nulo_estructura_oe4.csv'),
                                            on=['L', 'replicas', 'objeto', 'nw', 'pesos'], suffixes=('', '_r'))
    dif_oe = max(float(np.max(np.abs(oe[c] - oe[c + '_r']))) for c in ['prop_obs', 'prop_nulo_p50', 'prop_nulo_p95', 'p_valor'])
    print(f'{via}: k* distintos {n_dif} de {len(m)}; dif. máx. RV {dif_rv:.1e} ({len(pr)} pares); resúmenes {dif_rs:.1e}; '
          f'p5 por posición {dif_p5:.1e} ({len(po)} pares, marcas distintas {n_marca}); OE4 {dif_oe:.1e} ({len(oe)} filas)')
    assert n_dif == 0 and dif_rv < 1e-10 and dif_rs < 1e-10 and dif_p5 < 1e-10 and n_marca == 0 and dif_oe < 1e-12, via
    assert len(po) > 0 and len(oe) == 3 * len(OBJETOS) * len(ne.NW_GRILLA) * len(ne.PESOS)

niveles: k* distintos 0 de 1464; dif. máx. RV 7.7e-15 (48720 pares); resúmenes 1.4e-15; p5 por posición 2.0e-15 (6960 pares, marcas distintas 0); OE4 5.0e-16 (48 filas)


## 4. Lectura de D10, D13 y D17.3 (niveles)

- **Candidatos.** Regla del 06 (PC2, paso 3; sin pesos: se lee con los pares de pesos uniformes,
  D17.3) y `aede` (codo por caída máxima sobre λ1, maxcl = 10, D17.1), con los dos esquemas de pesos.
  Fechas: centro de ventana.
- **Borde de ventana** (D7): la transición está a ±3 meses (±6 como sensibilidad) del centro en que
  un episodio entra o sale de la ventana (`src/episodios.borde_ventana`).
- **Fuera del nulo** (D10, D17.3): algún par adyacente del mismo ancho y pesos, con frontera a
  ±3 meses de la transición, tiene un RV observado (cargas sin centrar) bajo su p5 por posición
  ($L = 24$, 200 réplicas). Definición operativa declarada en `src/dinamica.py`.
- **2017-10** (D13 con D17.3): confirmada solo si hay candidatos a ±3 meses que no son borde y quedan
  fuera del nulo con al menos dos anchos (regla del 06 o `aede` uniforme) **y** al menos uno en
  `aede` gaussiano; si no, «transición no confirmada». La sensibilidad repite la clasificación con
  el RV de las matrices de correlación completas.

In [5]:
if 'niveles' in VIAS:
    M, d, dr = VIAS['niveles']
    EPISODIOS = cargar_episodios(RAIZ)
    bordes_ventana(EPISODIOS, ne.NW_GRILLA).to_csv(d / '07b_bordes_ventana.csv', index=False)
    t06 = pd.read_csv(d / '06_epca_transiciones_ventanas_niveles.csv')
    cand = [pd.DataFrame({'metodo': 'regla 06 (PC2)', 'nw': t06.ventana, 'pesos': 'uniformes',
                          'fecha': t06.fecha_transicion_candidata.str[:7]})]
    ac = pd.read_csv(d / '06c_aede_cortes.csv', keep_default_na=False)
    ac = ac[(ac.maxcl == MAXCL_AEDE) & (ac.criterio == 'codo (caída máxima)')]
    for r in ac.itertuples():
        cand.append(pd.DataFrame({'metodo': 'aede codo', 'nw': r.nw, 'pesos': r.pesos, 'fecha': r.cortes.split()}))
    cand = pd.concat(cand, ignore_index=True)
    pos_all = pd.read_csv(d / '07b_nulo_estructura_rv_posicion.csv')
    pos_all = pos_all[(pos_all.L == 24) & (pos_all.replicas == 200)]
    marcas, tablas, resumen = [], [], []
    for obj in OBJETOS:
        m = dn.marcar_transiciones(cand, pos_all[pos_all.objeto == obj], EPISODIOS).assign(objeto=obj)
        t, res = dn.tabla_2017_10(m)
        marcas.append(m)
        tablas.append(t.assign(objeto=obj))
        resumen.append({'objeto': obj, 'principal': obj == dn.OBJETO_PRINCIPAL, **res})
    marcas = pd.concat(marcas, ignore_index=True)
    cols = ['objeto', 'metodo', 'nw', 'pesos', 'fecha', 'borde_3', 'es_borde_3', 'borde_6', 'es_borde_6',
            'pares_cercanos', 'pares_cercanos_bajo_p5', 'fuera_del_nulo']
    marcas[cols].to_csv(d / '07b_transiciones_d10.csv', index=False)
    tablas = pd.concat(tablas, ignore_index=True)
    tablas[['objeto'] + [c for c in tablas.columns if c != 'objeto']].to_csv(d / '07b_tabla_2017_10.csv', index=False)
    resumen = pd.DataFrame(resumen)
    resumen.to_csv(d / '07b_clasificacion_2017_10.csv', index=False)
    display(marcas[marcas.objeto == dn.OBJETO_PRINCIPAL][cols[1:]])
    display(tablas[tablas.objeto == dn.OBJETO_PRINCIPAL])
    display(resumen)

,metodo,nw,pesos,fecha,borde_3,es_borde_3,borde_6,es_borde_6,pares_cercanos,pares_cercanos_bajo_p5,fuera_del_nulo
0,regla 06 (PC2),24,uniformes,2008-07,—,False,—,False,0,0,False
1,regla 06 (PC2),24,uniformes,2009-10,—,False,—,False,7,0,False
2,regla 06 (PC2),24,uniformes,2012-07,Entra Taper tantrum,True,Entra Taper tantrum,True,7,0,False
3,regla 06 (PC2),24,uniformes,2013-07,Entra Colapso de las materias primas,True,Entra Colapso de las materias primas,True,7,0,False
4,regla 06 (PC2),24,uniformes,2015-01,Sale Taper tantrum,True,Sale Taper tantrum,True,7,0,False
5,regla 06 (PC2),24,uniformes,2017-01,Entra Kuczynski / Odebrecht; Sale Colapso de l...,True,Entra Kuczynski / Odebrecht; Entra Guerra come...,True,7,0,False
6,regla 06 (PC2),24,uniformes,2024-07,—,False,—,False,0,0,False
7,regla 06 (PC2),36,uniformes,2009-10,—,False,—,False,1,0,False
8,regla 06 (PC2),36,uniformes,2011-04,—,False,Sale Crisis financiera global,True,7,0,False
9,regla 06 (PC2),36,uniformes,2012-07,—,False,Entra Colapso de las materias primas,True,7,4,True


,metodo,nw,pesos,candidatos,candidato,algun_no_borde,algun_fuera_del_nulo,cuenta,objeto
0,regla 06 (PC2),24,uniformes,,False,False,False,False,cargas_sin_centrar
1,regla 06 (PC2),36,uniformes,,False,False,False,False,cargas_sin_centrar
2,regla 06 (PC2),48,uniformes,,False,False,False,False,cargas_sin_centrar
3,regla 06 (PC2),60,uniformes,2017-10 2018-01,True,False,False,False,cargas_sin_centrar
4,aede codo,24,uniformes,,False,False,False,False,cargas_sin_centrar
5,aede codo,36,uniformes,,False,False,False,False,cargas_sin_centrar
6,aede codo,48,uniformes,,False,False,False,False,cargas_sin_centrar
7,aede codo,60,uniformes,,False,False,False,False,cargas_sin_centrar
8,aede codo,24,gaussianos_nsd3,,False,False,False,False,cargas_sin_centrar
9,aede codo,36,gaussianos_nsd3,,False,False,False,False,cargas_sin_centrar


,objeto,principal,objetivo,anchos_que_cuentan_06_o_aede_uniformes,condicion_dos_anchos,anchos_que_cuentan_aede_gaussianos,condicion_gaussianos,clasificacion
0,cargas_sin_centrar,True,2017-10,,False,,False,transición no confirmada
1,correlacion,False,2017-10,,False,,False,transición no confirmada


**Lectura.** Sin interpretación en este notebook (regla 7): las cifras se leen en la nota del
desarrollador y en la sección 4 del Rmd final, con Roberto y el Dr. Camiz.